<a href="https://colab.research.google.com/github/suneel-ratnala/Agentic_AI_Practice/blob/main/day2/Lab2_2_RAG_Conversational.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 1.2: Conversational RAG - Mortgage Policy Assistant

In this lab, we will build a **Banking Policy Assistant** for Wells Fargo. This assistant allows loan officers to query internal mortgage guidelines (e.g., Conforming vs. Jumbo loans) and ask follow-up questions while maintaining conversational context.

## Use Case
A loan officer needs to quickly check credit score requirements for different loan types without searching through a 200-page PDF manual. They might ask:
1. "What is the min credit score for a Jumbo loan?"
2. "How does that compare to FHA?" (Contextual follow-up)

## Key Concepts
1. **History Aware Retriever**: Rephrases the user's latest query using the chat history so it makes sense as a standalone query to the vector store.
2. **Chat History**: Maintaining line of conversation.

In [1]:
# 1. Install Dependencies
print("Installing dependencies...")
%pip install -qU langchain langchain-groq langchain-community langchain-huggingface chromadb sentence-transformers
print("Dependencies installed.")

Installing dependencies...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.4/112.4 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 57.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 74.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.4/512.4 kB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.5/503.5 kB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.5/167.5 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 78.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━

In [2]:
# 2. Setup API Keys
from google.colab import userdata
import os

os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
# Optional for LangSmith Tracking
os.environ["LANGSMITH_API_KEY"] = userdata.get('LANGSMITH_API_KEY')
os.environ["LANGSMITH_TRACING_V2"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGSMITH_PROJECT"] = "RAG Conversational"

In [3]:
banking_policy_text = """
Wells Fargo Mortgage Lending Guidelines - Internal Policy Use Only

1. CONFORMING LOANS
   - Definition: Loans that meet the purchase limits set by Fannie Mae and Freddie Mac.
   - Max Loan Amount: $766,550 for single-family homes (as of 2024).
   - Minimum Credit Score: 620.
   - Down Payment: Minimum 3% for first-time homebuyers, otherwise 5%.
   - DTI Ratio: Maximum 50% with compensating factors.
   - Reserves: Not always required, depending on DU/LP findings.

2. JUMBO LOANS
   - Definition: Loans exceeding the conforming loan limits.
   - Minimum Credit Score: 700 for LTV up to 80%; 720 for LTV up to 90%.
   - Down Payment: Minimum 10% required.
   - DTI Ratio: Strictly capped at 43%.
   - Reserves: 6 months of PITI required.
   - Appraisal: Two full appraisals required for loan amounts > $1.5M.

3. FHA LOANS
   - Definition: Government-backed loans insured by the Federal Housing Administration.
   - Minimum Credit Score: 580 for 3.5% down payment; 500-579 for 10% down payment.
   - DTI Ratio: Up to 57% allowed in some cases.
   - MIP (Mortgage Insurance Premium): Upfront 1.75% + Annual MIP required for the life of the loan if LTV > 90%.
   - Property Requirements: Must meet FHA safety guidelines (no peeling paint, safety handrails required).

4. VA LOANS
   - Definition: Loans for eligible veterans and service members.
   - Down Payment: 0% required.
   - PMI: No private mortgage insurance required.
   - Funding Fee: Required unless the veteran has a service-connected disability.
   - DTI Ratio: No strict cap, but residual income analysis is key.

5. GENERAL UNDERWRITING REQUIREMENTS
   - Income: 2 years of consistent employment history required. Self-employed borrowers need 2 years of tax returns.
   - Assets: Large deposits must be sourced and explained.
   - Bankruptcy:
     - Chapter 7: 4-year waiting period for Conventional, 2 years for FHA/VA.
     - Chapter 13: 2-year waiting period after discharge for Conventional, 1 year of payout period for FHA/VA.
"""

In [4]:
# 3. Setup Vector Store (Mortgage Policy Data)
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document

# Load Internal Policy Document
print("Loading mortgage policy document...")
docs = [Document(page_content=banking_policy_text, metadata={"source": "internal_policy_doc"})]
print(f"Loaded {len(docs)} document(s).")

# Split
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
splits = text_splitter.split_documents(docs)
print(f"Split into {len(splits)} chunks.")

# Embed & Store
print("Creating vector store...")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(documents=splits, embedding=embeddings)
retriever = vectorstore.as_retriever()
print("Vector store created.")

Loading mortgage policy document...
Loaded 1 document(s).
Split into 5 chunks.
Creating vector store...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector store created.


## 4. History Aware Retriever
We need a chain that takes the `chat_history` and the `input` and generates a search query.

In [5]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Initialize LLM
llm = ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0,
    reasoning_format="parsed"
)

contextualize_q_system_prompt = (
    "Given a chat history and the latest user question "
    "which might reference context in the chat history, "
    "formulate a standalone question which can be understood "
    "without the chat history. Do NOT answer the question, "
    "just reformulate it if needed and otherwise return it as is."
)

contextualize_q_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", contextualize_q_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

# Chain to rephrase question
contextualize_q_chain = contextualize_q_prompt | llm | StrOutputParser()
print("Contextualization chain created.")

Contextualization chain created.


In [6]:
# Let's test the contextualization to understand what it does
from langchain_core.messages import HumanMessage, AIMessage

print("--- Testing Query Contextualization ---")
# Mock history: user asked about Jumbo loans, AI answered.
sample_history = [
    HumanMessage(content="What is the min credit score for a Jumbo Loan?"),
    AIMessage(content="The minimum credit score is 700.")
]
# User asks a follow-up specific to the context (referring to 'that')
sample_input = "How does that compare to FHA?"
print(f"Chat History: {len(sample_history)} messages")
print(f"User Follow-up: {sample_input}")

rephrased_query = contextualize_q_chain.invoke({"chat_history": sample_history, "input": sample_input})
print(f"Rephrased Query (for Vector Store): {rephrased_query}")
print("---------------------------------------")

--- Testing Query Contextualization ---
Chat History: 2 messages
User Follow-up: How does that compare to FHA?
Rephrased Query (for Vector Store): What is the minimum credit score required for an FHA loan compared to a Jumbo Loan?
---------------------------------------


## 5. QA Chain with History
Now we create the final chain that uses the retrieved documents to answer.

In [7]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

qa_system_prompt = (
    "You are a specialized Mortgage Policy Assistant for Wells Fargo. "
    "Use the following pieces of retrieved policy context to answer "
    "the loan officer's question. If you don't know the answer, say that you "
    "cannot find it in the policy. Keep answers professional and concise."
    "\n\n"
    "{context}"
)

qa_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", qa_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

def contextualized_question(input: dict):
    if input.get("chat_history"):
        return contextualize_q_chain
    else:
        return input.get("input")

rag_chain = (
    RunnablePassthrough.assign(
        context=contextualized_question | retriever | format_docs
    )
    | qa_prompt
    | llm
    | StrOutputParser()
)
print("Full RAG chain created.")

Full RAG chain created.


## 6. Testing the Chat
We can now manage specific chat sessions.

In [8]:
chat_history = []

# First Question: Specific Policy Query
user_input = "What is the minimum credit score for a Jumbo Loan?"

print(f"User: {user_input}")
# rag_chain returns string now
answer = rag_chain.invoke({"input": user_input, "chat_history": chat_history})
print(f"AI: {answer}")

# Update History
chat_history.extend([HumanMessage(content=user_input), AIMessage(content=answer)])

# Second Question (Follow-up): Contextual Comparison
print("\n--- Follow up ---")
user_input = "How does that compare to FHA loans?"
print(f"User: {user_input}")

answer = rag_chain.invoke({"input": user_input, "chat_history": chat_history})
print(f"AI: {answer}")

chat_history.extend([HumanMessage(content=user_input), AIMessage(content=answer)])

User: What is the minimum credit score for a Jumbo Loan?
AI: The minimum credit score for a Jumbo Loan is **700 for a loan-to-value (LTV) ratio up to 80%** and **720 for an LTV ratio up to 90%**. These requirements are outlined in the Wells Fargo Mortgage Lending Guidelines for Jumbo Loans.

--- Follow up ---
User: How does that compare to FHA loans?
AI: The minimum credit score for **FHA loans** is **580** for a 3.5% down payment or **500–579** for a 10% down payment, which is significantly lower than the **700–720** required for jumbo loans. This reflects the higher risk tolerance for government-backed FHA loans compared to jumbo loans, which are typically for higher-value properties and require stricter credit standards.


In [9]:
# Third Question (Follow-up): Contextual Comparison
print("\n--- Follow up ---")
user_input = "How does those two are different from VA loans?"
print(f"User: {user_input}")

answer = rag_chain.invoke({"input": user_input, "chat_history": chat_history})
print(f"AI: {answer}")

chat_history.extend([HumanMessage(content=user_input), AIMessage(content=answer)])


--- Follow up ---
User: How does those two are different from VA loans?
AI: VA loans differ from both Jumbo and FHA loans in several key ways:  

1. **Down Payment**:  
   - **VA Loans**: 0% required (no down payment).  
   - **Jumbo Loans**: Minimum 10% down.  
   - **FHA Loans**: 3.5% down (with 580+ credit score) or 10% down (with 500–579 credit score).  

2. **Credit Score Requirements**:  
   - **VA Loans**: No strict minimum credit score (underwriters use discretion).  
   - **Jumbo Loans**: Minimum 700 (LTV ≤80%) or 720 (LTV ≤90%).  
   - **FHA Loans**: Minimum 580 (3.5% down) or 500–579 (10% down).  

3. **Insurance/Fees**:  
   - **VA Loans**: No PMI; a **funding fee** is required (waived for veterans with service-connected disabilities).  
   - **Jumbo Loans**: No PMI, but stricter DTI (43% cap) and reserves (6 months of PITI).  
   - **FHA Loans**: Upfront MIP (1.75%) + annual MIP (for life if LTV >90%).  

4. **DTI Ratio**:  
   - **VA Loans**: No strict cap, but **residua